# FinanceToolkit Computation Library EDA

FinanceToolkit is best treated as an FMP-derived computation library for financial statement ratios, growth, and analytics. This notebook demonstrates the intended integration boundary without forcing those semantics onto other vendors.

In [1]:
from __future__ import annotations

import importlib
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

def required_library_status(module_name: str) -> pd.DataFrame:
    try:
        module = importlib.import_module(module_name)
        return pd.DataFrame([{
            "module": module_name,
            "required": True,
            "installed": True,
            "version": getattr(module, "__version__", "unknown"),
            "module_file": getattr(module, "__file__", None),
        }])
    except Exception as exc:
        return pd.DataFrame([{
            "module": module_name,
            "required": True,
            "installed": False,
            "version": None,
            "module_file": None,
            "error": f"{type(exc).__name__}: {exc}",
        }])

def synthetic_prices(rows: int = 360, symbol: str = "AAPL") -> pd.DataFrame:
    rng = np.random.default_rng(7)
    dates = pd.date_range("2024-01-02", periods=rows, freq="B")
    drift = 0.00055
    shock = rng.normal(0, 0.015, rows)
    close = 100 * np.exp(np.cumsum(drift + shock))
    open_ = close * (1 + rng.normal(0, 0.003, rows))
    high = np.maximum(open_, close) * (1 + rng.uniform(0.001, 0.02, rows))
    low = np.minimum(open_, close) * (1 - rng.uniform(0.001, 0.02, rows))
    volume = rng.integers(900_000, 4_500_000, rows).astype(float)
    return pd.DataFrame({
        "symbol": symbol,
        "date": dates,
        "open": open_,
        "high": high,
        "low": low,
        "close": close,
        "volume": volume,
    })

prices = synthetic_prices()
prices.head()

,symbol,date,open,high,low,close,volume
0,AAPL,2024-01-02,100.2080,101.2355,98.8738,100.0569,"4,479,129.0000"
1,AAPL,2024-01-03,101.1259,102.2340,100.4497,100.5615,"2,365,734.0000"
2,AAPL,2024-01-04,100.3819,101.9042,99.3952,100.2040,"1,409,205.0000"
3,AAPL,2024-01-05,98.9452,99.8670,97.0239,98.9286,"4,225,356.0000"
4,AAPL,2024-01-08,97.8130,98.6722,97.4943,98.3103,"2,886,674.0000"


## Required Dependency Status

In [2]:
required_library_status("financetoolkit")

,module,required,installed,version,module_file
0,financetoolkit,True,True,unknown,/home/jlee153232/miniconda3/lib/python3.13/sit...


## Synthetic FMP-Like Fundamentals

FinanceToolkit-style features belong near FMP-derived feature storage because the library expects FMP-like data semantics. The example below mimics statement-level inputs and derives reusable ratio features.

In [3]:
periods = pd.period_range("2022Q1", periods=12, freq="Q")
fundamentals = pd.DataFrame({
    "date": periods.to_timestamp("Q"),
    "symbol": "AAPL",
    "revenue": np.linspace(90_000, 125_000, len(periods)) + np.random.default_rng(3).normal(0, 3_000, len(periods)),
    "gross_profit": np.linspace(38_000, 58_000, len(periods)),
    "operating_income": np.linspace(24_000, 42_000, len(periods)),
    "net_income": np.linspace(20_000, 37_000, len(periods)),
    "total_assets": np.linspace(340_000, 390_000, len(periods)),
    "total_equity": np.linspace(62_000, 83_000, len(periods)),
    "free_cash_flow": np.linspace(19_000, 34_000, len(periods)),
})
fundamentals.tail()

,date,symbol,revenue,gross_profit,operating_income,net_income,total_assets,total_equity,free_cash_flow
7,2023-12-31,AAPL,"111,576.9301","50,727.2727","35,454.5455","30,818.1818","371,818.1818","75,363.6364","28,545.4545"
8,2024-03-31,AAPL,"112,858.9062","52,545.4545","37,090.9091","32,363.6364","376,363.6364","77,272.7273","29,909.0909"
9,2024-06-30,AAPL,"128,605.3622","54,363.6364","38,727.2727","33,909.0909","380,909.0909","79,181.8182","31,272.7273"
10,2024-09-30,AAPL,"122,495.5417","56,181.8182","40,363.6364","35,454.5455","385,454.5455","81,090.9091","32,636.3636"
11,2024-12-31,AAPL,"123,942.1076","58,000.0000","42,000.0000","37,000.0000","390,000.0000","83,000.0000","34,000.0000"


In [4]:
ft_features = fundamentals.copy()
ft_features["ft__gross_margin"] = ft_features["gross_profit"] / ft_features["revenue"]
ft_features["ft__operating_margin"] = ft_features["operating_income"] / ft_features["revenue"]
ft_features["ft__net_margin"] = ft_features["net_income"] / ft_features["revenue"]
ft_features["ft__return_on_equity"] = ft_features["net_income"] / ft_features["total_equity"]
ft_features["ft__return_on_assets"] = ft_features["net_income"] / ft_features["total_assets"]
ft_features["ft__fcf_margin"] = ft_features["free_cash_flow"] / ft_features["revenue"]
ft_features["ft__revenue_growth_qoq"] = ft_features["revenue"].pct_change()
ft_features[["date", "symbol", *[c for c in ft_features.columns if c.startswith("ft__")]]].tail(8)

,date,symbol,ft__gross_margin,ft__operating_margin,ft__net_margin,ft__return_on_equity,ft__return_on_assets,ft__fcf_margin,ft__revenue_growth_qoq
4,2023-03-31,AAPL,0.4466,0.3013,0.2583,0.3760,0.0731,0.2412,0.0360
5,2023-06-30,AAPL,0.4474,0.3057,0.2634,0.3875,0.0764,0.2453,0.0384
6,2023-09-30,AAPL,0.4747,0.3282,0.2841,0.3985,0.0797,0.2638,-0.0212
7,2023-12-31,AAPL,0.4546,0.3178,0.2762,0.4089,0.0829,0.2558,0.0829
8,2024-03-31,AAPL,0.4656,0.3286,0.2868,0.4188,0.0860,0.2650,0.0115
9,2024-06-30,AAPL,0.4227,0.3011,0.2637,0.4282,0.0890,0.2432,0.1395
10,2024-09-30,AAPL,0.4586,0.3295,0.2894,0.4372,0.0920,0.2664,-0.0475
11,2024-12-31,AAPL,0.4680,0.3389,0.2985,0.4458,0.0949,0.2743,0.0118


## Feature Engineering Connection

Existing Quant Warehouse fundamental feature modules already broadcast point-in-time statement data to daily rows. FinanceToolkit-derived outputs should follow the same pattern: compute once from FMP semantics, prefix clearly, then broadcast or store as reusable features.

In [5]:
daily_index = pd.MultiIndex.from_product(
    [pd.date_range("2024-01-01", periods=10, freq="B"), ["AAPL"]],
    names=["date", "symbol"],
)
quarterly = ft_features.set_index(["date", "symbol"])[[c for c in ft_features.columns if c.startswith("ft__")]]
# Point-in-time sketch: as-of join quarterly values onto daily target dates.
daily = pd.DataFrame(index=daily_index).reset_index()
q = quarterly.reset_index().sort_values("date")
daily_features = pd.merge_asof(
    daily.sort_values("date"),
    q.sort_values("date"),
    by="symbol",
    on="date",
    direction="backward",
).set_index(["date", "symbol"])
daily_features.tail()

,,ft__gross_margin,ft__operating_margin,ft__net_margin,ft__return_on_equity,ft__return_on_assets,ft__fcf_margin,ft__revenue_growth_qoq
date,symbol,,,,,,,
2024-01-08,AAPL,0.4546,0.3178,0.2762,0.4089,0.0829,0.2558,0.0829
2024-01-09,AAPL,0.4546,0.3178,0.2762,0.4089,0.0829,0.2558,0.0829
2024-01-10,AAPL,0.4546,0.3178,0.2762,0.4089,0.0829,0.2558,0.0829
2024-01-11,AAPL,0.4546,0.3178,0.2762,0.4089,0.0829,0.2558,0.0829
2024-01-12,AAPL,0.4546,0.3178,0.2762,0.4089,0.0829,0.2558,0.0829


## Target Engineering Connection

FinanceToolkit outputs are features. Pair them with targets like forward returns, cross-sectional ranks, or event-pair labels. Do not put strategy or model-selection logic in Quant Warehouse.

In [6]:
from quant_warehouse.platforms.data_providers.fmp.target_engineering import build_forward_return_labels

labels = build_forward_return_labels(prices[["symbol", "date", "close"]], horizons=[20, 60])
labels.head()

,symbol,date,horizon,target_name,target_value
0,AAPL,2024-01-02,20,forward_return_20d,-0.1056
1,AAPL,2024-01-02,60,forward_return_60d,-0.1413
2,AAPL,2024-01-03,20,forward_return_20d,-0.1127
3,AAPL,2024-01-03,60,forward_return_60d,-0.1511
4,AAPL,2024-01-04,20,forward_return_20d,-0.1258


## Notes

- Keep FinanceToolkit results under FMP-derived feature namespaces unless an equivalent implementation is built for another vendor.
- Good candidates: margins, returns on capital, growth, leverage, valuation, quality, and Piotroski/Altman-style composites.
- Validate against existing `feature_engineering.fundamental_features` before promoting into production feature generation.